# Week 2 Day 5

## Covered Today
1. Introduction to Agentic AI and Building Multi-Tool Workflows
2. How Gradio Works: Building Web UIs from Python Code
3. Building Multi-Modal Apps with DALL-E 3, Text-to-Speech, and Gradio Bloc
4. Running Your Multimodal AI Assistant with Gradio and Tools
5. Extra - Compare Frontier LLMs with OpenRouter: Generate SVG Art in Python

In [4]:
# we start by setting up the entire boilerplate for AI LLM interacion


# required imports
import os
from dotenv import load_dotenv
import requests
from openai import OpenAI
from IPython.display import Markdown, display, update_display
import gradio as gr 
import json


# load API Keys
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')


# check if all loaded keys exist and are as per the format
if openai_api_key:
    if openai_api_key.startswith("sk-"):
        print(f"OpenAI      : OK           (begins {openai_api_key[:7]}...)")
    else:
        print("OpenAI      : WRONG FORMAT (should start with 'sk-')")
else:
    print("OpenAI      : MISSING")

if anthropic_api_key:
    if anthropic_api_key.startswith("sk-ant-"):
        print(f"Anthropic   : OK           (begins {anthropic_api_key[:10]}...)")
    else:
        print("Anthropic   : WRONG FORMAT (should start with 'sk-ant-')")
else:
    print("Anthropic   : MISSING")

if google_api_key:
    if google_api_key.startswith("AQ.Ab"):
        print(f"Google      : OK           (begins {google_api_key[:5]}...)")
    else:
        print("Google      : WRONG FORMAT (should start with 'AQ.Ab' or 'AIza')")
else:
    print("Google      : MISSING")

if openrouter_api_key:
    if openrouter_api_key.startswith("sk-or-"):
        print(f"OpenRouter  : OK           (begins {openrouter_api_key[:8]}...)")
    else:
        print("OpenRouter  : WRONG FORMAT (should start with 'sk-or-')")
else:
    print("OpenRouter  : MISSING")


# create clients for each provider
openai_client = OpenAI()
google_url = 'https://generativelanguage.googleapis.com/v1beta/openai/'
anthropic_url = 'https://api.anthropic.com/v1/'
openrouter_url = 'https://openrouter.ai/api/v1'
ollama_url = 'http://127.0.0.1:11434/v1'


google_client = OpenAI(base_url=google_url, api_key=google_api_key)
anthropic_client = OpenAI(base_url=anthropic_url, api_key=anthropic_api_key)
openrouter_client = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama_client = OpenAI(base_url=ollama_url, api_key='Ollama')

OpenAI      : OK           (begins sk-proj...)
Anthropic   : OK           (begins sk-ant-api...)
Google      : OK           (begins AQ.Ab...)
OpenRouter  : OK           (begins sk-or-v1...)


In [5]:
# since, Ed is using sqlite3 to host info on day 5, we also practice it side by side, so as to reinforce the learning. Sqlite is used in cases where either we have a small database requirement or no complications of setting up a server is needed. Sqlite3 is a db library, that ships with python, so it is super easy to create a db within your program for experimental work. To start, we start by importing the library.

import sqlite3

In [ ]:
# now, import is done. we need to connect to the db file. Now, an sqlite db file is binary, hence, opening it in our IDE will only show gibberish. To connect our python code to the db, we do the following. This is a version that is being used for understanding. Later, we will further refine this code. For now, in the dir, we do not have any db file. 

# conn = sqlite3.connect('tickets.db')

In [ ]:
# as soon as we run the above, if the file exists, our python code makes a connection with it to send or receive data, and if the file does not exist, then it is created first and then the connection is made, by this one command.

# Now, even though the db has been created and connection has been made. Now, we need a way to pass through this connection and carry data both ways and this much happen simultaneously as our program may call multiple queries. For this, sqlite3 provides a method cursor. We simply create an instance by doing:
# cursor = conn.cursor()
# Now, by using cursor, we can interact with our data(manipulate it) which will be using the connection conn.

In [ ]:
# Sqlite cursor has two types of commands primarily (needed for us now): execute: this is used to execute sql commands and send data to the db, while fetch is used to read or get data from the db. Now, we use execute to create a new table in our db. For this, we use the following:

# The command in sql is simple: CREATE TABLE table_name(name datatype), this is exactly what we also use here, just by using execute and the command goes in as a string:

# cursor.execute("CREATE TABLE prices(city TEXT, price INTEGER)")

In [ ]:
# Now, we need to also save the changes. For it, we use commit method, not on the cursor, but on the connection.
# conn.commit()

In [ ]:
# Now, our table is saved. Next, we add data to this table. The syntax is very simple, just like normal SQL and adding cursor.execute to it.

# cursor.execute("INSERT INTO prices VALUES('london', '1299')")

In [ ]:
# like last time, we again need to save this
# conn.commit()

In [ ]:
# Now we move ahead with reading of the information. For this, we need to follow a two step approach, first we execute a select command and then we run a fetch command. lets first execute a select command
# cursor.execute("SELECT * FROM prices")

In [ ]:
# Now, to view the results, we need to use fetch command. fetch has multiple variations, we for now run, fetchall, to view the entire table for now, since we have selected the entire table also in the last command. 

# cursor.fetchall()

[('london', 1299)]

In [ ]:
# here the result gives us only one row, since we have only one row in our data. Point to notice here is that for a row, we have received the data inside a tuple, which is further inside a list. Let's add some more data, to learn more about fetch.

# cursor.execute("INSERT INTO prices VALUES('mumbai', 399), ('chennai', 299), ('rome', 999)")
# conn.commit()

In [ ]:
# now, lets read again
# cursor.execute("SELECT * FROM prices")
# cursor.fetchall()

[('london', 1299), ('mumbai', 399), ('chennai', 299), ('rome', 999)]

In [ ]:
# Now, we have the table, with 4 tuples inside a list. Now, instead of using fetchall, we can also use fetchone
# cursor.execute("SELECT * FROM prices")
# cursor.fetchone()

# # This will give us the first row only.

('london', 1299)

In [ ]:
# cursor.fetchone()


('mumbai', 399)

In [ ]:
# cursor.fetchone()


('chennai', 299)

In [ ]:
# This is to show that everytime we now use fetch one after executing select, we get each row, one after the other. Also, in addition to fetchall and fetchone, we have two more ways to fetch this information. also, if we keep doing fetchone, at the end it will return None. Next is fetchmany, here, we define the number of rows we want to see.

# cursor.execute("SELECT * FROM prices")
# cursor.fetchmany(2)

[('london', 1299), ('mumbai', 399)]

In [ ]:
# cursor.fetchmany(3)

[('chennai', 299), ('rome', 999)]

In [ ]:
# cursor.fetchmany(2)

[]

In [ ]:
# The behaviour is very similar to fetchone, a subsequent command in fetchmany will also return the next set of rows, ultimately reaching None.
# Next is, using for loop. 

# for row in cursor.execute("SELECT * FROM prices"):
#     print(row)

# # the entire table is iterated over and we get individual tuples

('london', 1299)
('mumbai', 399)
('chennai', 299)
('rome', 999)


In [ ]:
# Also, when using user inputs to insert values in the data, we might think of using f strings, something like this:
# let's say we are asking the user to add more data to our table, then, one way can be:
# cursor.execute(f"INSERT INTO prices VALUES('{city}', {price})"), here, if we see, we have '' outside city, since it is a string. Also, if a user enters DELETE FROM TABLE, as SQL injection, then there can be a risk of that command running. However when we use the below format, in that case, each of the added values is marked as a variable by sqlite 3, and stored in the DB. Therefore, to prevent quote juggling and also security or SQL injection, we use the following format:

# city = 'new york'
# price = 1499
# cursor.execute("INSERT INTO prices VALUES(?, ?)", (city, price))
# # here ? acts as a placeholder

In [ ]:
# conn.commit()

# for row in cursor.execute("SELECT * FROM prices"):
#     print(row)

('london', 1299)
('mumbai', 399)
('chennai', 299)
('rome', 999)
('new york', 1499)


In [ ]:
# # now, we can see above, a new row has been added. Now, we also have with based context manager in SQL. Below is an example of the same
# price_list = {'london': '$1299', 'mumbai': '$399', 'chennai': '$299', 'rome': '$999', 'new york': '$1499'}
# with sqlite3.connect('prices_2.db') as connec:
#     cursor_new = connec.cursor()
#     cursor_new.execute("CREATE TABLE prices(city TEXT, prices TEXT)")
#     for city, price in price_list.items():
#         cursor_new.execute("INSERT INTO prices VALUES(?, ?)",(city, price))

# conn.close()

In [ ]:
# cursor_new.execute("SELECT * FROM prices")
# cursor_new.fetchall()

## Now we move to Day 5

In [6]:
system_message = "You are a helpful assistant for an airline called FlightAI. Give short, courteous answers, no more than one sentence. Always be accurate, if you do not know the answer, say so."


In [ ]:
# but before we go ahead, we need to setup our db in sqlite3 and then proceed ahead. 
flight_info = {'paris': '$1299', 'london': '$1399', 'mumbai': '$199', 'chennai': '$299', 'rome': '$999'}
with sqlite3.connect('flightai.db') as conn:
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE prices(city TEXT, price TEXT)")

conn.close()

In [66]:
with sqlite3.connect('flightai.db') as conn:
    cursor = conn.cursor()
    for city, price in flight_info.items():
        cursor.execute("INSERT INTO prices VALUES(?, ?)", (city, price))

    cursor.execute("SELECT * FROM prices")
    print(cursor.fetchall())

[('paris', '$1299'), ('london', '$1399'), ('mumbai', '$199'), ('chennai', '$299'), ('rome', '$999')]


In [8]:
# Now, since our DB is ready, we can proceed ahead.
conn.close()
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}")
    with sqlite3.connect('flightai.db') as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price FROM prices WHERE city = ?", (city.lower(), ))
        result = cursor.fetchone()

        return f"Ticket price to {city} is {result[0]}" if result else "No price data available for this city"

In [9]:
get_ticket_price('chennai')

DATABASE TOOL CALLED: Getting price for chennai


'Ticket price to chennai is $299'

In [10]:
# the function is working fine, now lets write the price function explaining the above function
price_function = {
    'name': 'get_ticket_price',
    'description': 'Fetches the price of return ticket to the destination city',
    'parameters': {
        'type': 'object',
        'properties': {
            'city': {
                'type': 'string',
                'description': 'The city the customer wants to travel to'
            }
        },
        'required': ['city'],
        'additionalProperties': False
    }
}

In [11]:
tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Fetches the price of return ticket to the destination city',
   'parameters': {'type': 'object',
    'properties': {'city': {'type': 'string',
      'description': 'The city the customer wants to travel to'}},
    'required': ['city'],
    'additionalProperties': False}}}]

In [12]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai_client.chat.completions.create(model='gpt-4.1-mini', messages=messages, tools=tools)


    while response.choices[0].finish_reason == "tool_calls":
        llm_message = response.choices[0].message
        tool_response = handle_tool_calls(llm_message)
        messages.append(llm_message)
        messages.extend(tool_response)
        response = openai_client.chat.completions.create(model='gpt-4.1-mini', messages=messages, tools=tools)

    
    return response.choices[0].message.content

In [17]:
def handle_tool_calls(message):
    responses = []
    for call in message.tool_calls:
        if call.function.name == 'get_ticket_price':
            arguments = json.loads(call.function.arguments)
            city = arguments["city"]
            price = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price,
                "tool_call_id": call.id
            })
    return responses

In [ ]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for mumbai
DATABASE TOOL CALLED: Getting price for london
